In [1]:
import langchain
print(langchain.__file__)

C:\Users\szymo\.conda\envs\georag\lib\site-packages\langchain\__init__.py


In [2]:
import os
import sys
from dotenv import load_dotenv

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.document_loader import PDFDocumentLoader
from src.chunker import TextChunker
from src.retriever import VectorStoreManager
from src.generator import RAGGenerator
from src.evaluator import RAGEvaluator


load_dotenv(os.path.join('..', '.env'))
print("Środowisko załadowane.")

Środowisko załadowane.


### Wczytywanie i Chunkowanie

In [3]:
loader = PDFDocumentLoader(data_path="../data/raw")
documents = loader.load_documents()

chunker = TextChunker(chunk_size=1000, chunk_overlap=200)
chunked_docs = chunker.split_documents(documents)

2026-04-30 15:49:21,983 - INFO - Wczytywanie plików z katalogu: ../data/raw
2026-04-30 15:49:21,984 - INFO - Ładowanie: lidar_podstawy.pdf
2026-04-30 15:49:22,208 - INFO - Pomyślnie załadowano 1 stron z dokumentów.
2026-04-30 15:49:22,209 - INFO - Rozpoczynam podział 1 stron na mniejsze fragmenty...
2026-04-30 15:49:22,210 - INFO - Podzielono tekst na 2 chunków.


### Baza Wektorowa (Chroma)

In [4]:
vsm = VectorStoreManager(persist_directory="../data/processed/chroma_db")
db = vsm.build_database(chunked_docs)

2026-04-30 15:49:23,908 - INFO - Inicjalizacja modelu embeddingowego (może to zająć chwilę przy pierwszym uruchomieniu)...
2026-04-30 15:49:23,911 - INFO - No device provided, using cpu
2026-04-30 15:49:24,147 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-30 15:49:24,184 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/modules.json "HTTP/1.1 200 OK"
2026-04-30 15:49:24,327 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-30 15:49:24,328 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster download

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-04-30 15:49:25,945 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-04-30 15:49:26,089 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-04-30 15:49:26,234 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-04-30 15:49:26,386 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-04-30 15:49:26,535 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2

### Generacja odpowiedzi

In [5]:
generator = RAGGenerator(vector_store=db)

pytanie = "Jakie są główne zastosowania technologii LiDAR w geoinformatyce?"
odpowiedz = generator.ask(pytanie)

print(f"PYTANIE: {pytanie}\n")
print(f"ODPOWIEDŹ:\n{odpowiedz['answer']}\n")
print("ŹRÓDŁA:")
for doc in odpowiedz['context']:
    print(f"- Plik: {doc.metadata.get('source')}, Strona: {doc.metadata.get('page')}")

2026-04-30 15:49:32,336 - INFO - Inicjalizacja modelu LLM: gemini-2.5-flash (temperature=0.0)
2026-04-30 15:49:32,394 - INFO - Zadaję pytanie: 'Jakie są główne zastosowania technologii LiDAR w geoinformatyce?'
2026-04-30 15:49:32,435 - INFO - AFC is enabled with max remote calls: 10.
2026-04-30 15:49:34,043 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-30 15:49:34,057 - INFO - Odpowiedź wygenerowana pomyślnie. Znalezione źródła:
2026-04-30 15:49:34,059 - INFO -   [1] Plik: ../data/raw\lidar_podstawy.pdf, Strona: 0
2026-04-30 15:49:34,062 - INFO -   [2] Plik: ../data/raw\lidar_podstawy.pdf, Strona: 0
2026-04-30 15:49:34,063 - INFO -   [3] Plik: ../data/raw\lidar_podstawy.pdf, Strona: 0
2026-04-30 15:49:34,065 - INFO -   [4] Plik: ../data/raw\lidar_podstawy.pdf, Strona: 0


PYTANIE: Jakie są główne zastosowania technologii LiDAR w geoinformatyce?

ODPOWIEDŹ:
Główne zastosowania technologii LiDAR w geoinformatyce to:
*   Modelowanie zagrożeń powodziowych (tworzenie Numerycznego Modelu Terenu - NMT).
*   Inwentaryzacja lasów (obliczanie biomasy i wysokości drzew).
*   Projektowanie infrastruktury drogowej w koncepcji Smart City.

ŹRÓDŁA:
- Plik: ../data/raw\lidar_podstawy.pdf, Strona: 0
- Plik: ../data/raw\lidar_podstawy.pdf, Strona: 0
- Plik: ../data/raw\lidar_podstawy.pdf, Strona: 0
- Plik: ../data/raw\lidar_podstawy.pdf, Strona: 0


### Ewaluacja (LLM-as-a-Judge)

In [7]:
evaluator = RAGEvaluator()


context_text = "\n".join([doc.page_content for doc in odpowiedz['context']])

faithfulness = evaluator.evaluate_faithfulness(pytanie, context_text, odpowiedz['answer'])
relevance = evaluator.evaluate_relevance(pytanie, odpowiedz['answer'])

print("--- WYNIKI EWALUACJI ---")
print(f"Faithfulness (Wierność): {faithfulness['evaluation']}")
print(f"Answer Relevance (Trafność): {relevance['evaluation']}")

2026-04-30 15:50:38,542 - INFO - Inicjalizacja modułu Ewaluatora RAG...
2026-04-30 15:50:38,577 - INFO - Ocenianie wierności (Faithfulness)...
2026-04-30 15:50:38,579 - INFO - AFC is enabled with max remote calls: 10.
2026-04-30 15:50:41,532 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-30 15:50:41,534 - INFO - Ocenianie trafności (Answer Relevance)...
2026-04-30 15:50:41,536 - INFO - AFC is enabled with max remote calls: 10.
2026-04-30 15:50:44,782 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


--- WYNIKI EWALUACJI ---
Faithfulness (Wierność): WYNIK: 1
UZASADNIENIE: Wszystkie punkty wymienione w odpowiedzi są bezpośrednio i w pełni potwierdzone w ostatnim zdaniu kontekstu źródłowego, które wymienia główne zastosowania technologii LiDAR.
Answer Relevance (Trafność): WYNIK: 1
UZASADNIENIE: Odpowiedź bezpośrednio i precyzyjnie adresuje intencję pytania, wymieniając konkretne zastosowania technologii LiDAR w geoinformatyce.
